Notebook to generate the ranking based on the ratio between the model and baseline WIS

In [1]:
import numpy as np 
import pandas as pd 
from aux_func import code_to_state, estado_para_regiao

In [2]:
challenge = 'dengue_state'

df_preds = pd.read_csv(f'./predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)
df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_1,id,validation,wis,model
0,2022-10-09,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
1,2022-10-16,37.019327,43.295517,51.743385,66.653189,83.207109,100.790847,116.992117,126.864602,134.406880,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
2,2022-10-23,55.945220,68.411390,84.200302,108.887826,135.698419,163.344639,189.017119,205.765306,218.575663,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
3,2022-10-30,58.185781,69.697014,83.792687,107.076328,132.601767,158.745145,184.276095,201.204977,212.912726,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
4,2022-11-06,58.634062,68.591352,80.855678,101.969117,125.548661,150.259612,174.517426,190.524410,202.512661,12,6822,1,95.67,3rd_imdc_isi_isi-dengue


In [3]:
df_agg_wis = (
    df_preds
    .groupby(["model", "adm_1", "validation"], as_index=False)["wis"]
    .mean()
)


df_agg_wis.head()

,model,adm_1,validation,wis
0,3rd_imdc_afya_ric,11,1,182.42
1,3rd_imdc_afya_ric,11,2,99.96
2,3rd_imdc_afya_ric,11,3,33.80
3,3rd_imdc_afya_ric,11,4,47.27
4,3rd_imdc_afya_ric,12,1,56.04


Gerando um ranking da média da diferença entre os modelos e o baseline: 

In [4]:
model_baseline = '3rd_imdc_procc_bb_model'

df_baseline = (
    df_agg_wis.loc[df_agg_wis["model"] == model_baseline,
           ["adm_1", "validation", "wis"]]
    .rename(columns={"wis": "wis_baseline"})
)

# Junta o WIS do baseline aos demais modelos
df_ratio = df_agg_wis.merge(
    df_baseline,
    on=["adm_1", "validation"],
    how="left"
)

# Razão WIS / Baseline
df_ratio["wis_ratio"] = (
    df_ratio["wis"] / df_ratio["wis_baseline"]
)

# Médias das razões
df_summary = (
    df_ratio
    .groupby(["adm_1", "model"], as_index=False)
    .agg(
        arithmetic_mean_ratio=("wis_ratio", "mean"),
        geometric_mean_ratio=("wis_ratio", lambda x: np.exp(np.mean(np.log(x)))),
    )
    .sort_values(["adm_1", "geometric_mean_ratio"])
)

df_summary.head()

,adm_1,model,arithmetic_mean_ratio,geometric_mean_ratio
5,11,3rd_imdc_emap_epidematicos_sarimax_state,0.803841,0.769598
19,11,3rd_imdc_pucrio_arbocaster,0.834013,0.789304
15,11,3rd_imdc_lncc_lncc_arp26_dengue,0.818201,0.800598
12,11,3rd_imdc_ifgw_inframind-proteus,0.964784,0.845450
22,11,3rd_imdc_rki_rki_zki_ph_lstm_geo,1.087766,0.900748


In [5]:
df_ratio['region'] = df_ratio['adm_1'].replace(code_to_state).replace(estado_para_regiao)

df_ratio.to_csv('predictions/rank_ratio.csv.gz', index = False)